In [119]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY') ## for LangSmith Tracking
os.environ['LANGSMITH_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [120]:
from langchain_groq import ChatGroq
llm_groq = ChatGroq(
  model= "Gemma2-9b-It"
)

## SystemMessage, HumanMessage, AIMessage:

In [121]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm_groq.invoke(
  [
    HumanMessage(content="Hi!!")
  ]
)

AIMessage(content='Hi there! 👋\n\nHow can I help you today? 😊\n', response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 11, 'total_tokens': 27, 'completion_time': 0.029090909, 'prompt_time': 0.00117225, 'queue_time': 0.252160649, 'total_time': 0.030263159}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-8ef8e8f0-ff16-40ce-9566-e1eb90646ad5-0', usage_metadata={'input_tokens': 11, 'output_tokens': 16, 'total_tokens': 27})

## Message Rememberance : Using List of messages : HumanMessage, AIMessage

In [122]:
llm_groq.invoke(
  [
      HumanMessage(content= "Hi!!"),
      AIMessage(content='Hi there! 👋  What can I do for you today? 😊\n'),
      HumanMessage(content= "What all did I ask you?")
  ]
)

AIMessage(content='You asked me "Hi!!".  😄  \n\nIs there anything else I can help you with? \n', response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 43, 'total_tokens': 68, 'completion_time': 0.045454545, 'prompt_time': 0.001643929, 'queue_time': 0.25215635, 'total_time': 0.047098474}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-610f74a7-1933-474b-92d9-472548506ab7-0', usage_metadata={'input_tokens': 43, 'output_tokens': 25, 'total_tokens': 68})

### Message History class: To wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load messages and pass them into tha chain as part of the input.

In [123]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}  ## storing chat history
def get_session_history(session_id: str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = ChatMessageHistory()  ## create new chat history for sessionId, if sessionId not present
  return store[session_id] ## return base chat history for this sessionId

with_message_history = RunnableWithMessageHistory(llm_groq, get_session_history) ## manage chat message history with LLM

In [124]:
config1 = {"configurable" : {"session_id" : "chat1"}}

In [125]:
response = with_message_history.invoke(
  [
    HumanMessage(content= "Hi! I am Ayush."),
  ],
  config= config1
)

In [126]:
print(response.content)

Hi Ayush, it's nice to meet you! 👋

How can I help you today?😊



In [127]:
response = with_message_history.invoke(
  [
    HumanMessage(content= "What is my name?"),
  ],
  config= config1
)
print(response.content)

Your name is Ayush, as you told me at the beginning. 😊  

Is there anything else I can help you with?



### Now if we change config:

In [128]:
config2 = {"configurable" : {"session_id" : "chat2"}}

response = with_message_history.invoke(
  [
    HumanMessage(content= "What is my name?"),
  ],
  config= config2
)
print(response.content)

As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to use it!



## Prompt Templates:

### NOTE : MessagesPlaceholder is meant for chat history (a list of messages), not for a single input string like a question.

In [129]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

prompt = ChatPromptTemplate.from_messages(
  [
    SystemMessagePromptTemplate.from_template("You are a helpful assistant. Answer all questions."),
    MessagesPlaceholder(variable_name= "message")
  ]
)

In [130]:
chain = prompt | llm_groq

In [131]:
chain.invoke(
  {"message" : [HumanMessage(content= "Hi! My name is Ayush")]}
)

AIMessage(content="Hello Ayush! 👋  \n\nIt's nice to meet you. How can I help you today? 😊  \n\n", response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 26, 'total_tokens': 54, 'completion_time': 0.050909091, 'prompt_time': 0.00140444, 'queue_time': 0.251955349, 'total_time': 0.052313531}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-3827c2f2-9ae1-46e8-8913-01e4981441b6-0', usage_metadata={'input_tokens': 26, 'output_tokens': 28, 'total_tokens': 54})

In [152]:
## Add more complexity
prompt = ChatPromptTemplate.from_messages(
  [
    SystemMessagePromptTemplate.from_template("You are a helpful assistant. Answer all questions in following language : {language}."),
    MessagesPlaceholder(variable_name= "message")
  ]
)

chain = prompt | llm_groq

In [133]:
response = chain.invoke(
    {"message" : [HumanMessage(content= "Hi! My name is Ayush")], "language": "Hindi"},
)

response

AIMessage(content='नमस्ते! मेरा नाम है बोत.  आप क्या पूछना चाहते हैं, आयुष? \n', response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 31, 'total_tokens': 61, 'completion_time': 0.054545455, 'prompt_time': 0.0014906, 'queue_time': 0.259320349, 'total_time': 0.056036055}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-ff9861c8-1658-456f-bda7-f5b2a6ae1781-0', usage_metadata={'input_tokens': 31, 'output_tokens': 30, 'total_tokens': 61})

### Now using this prompt with "with_message_history":

In [ ]:
with_message_history = RunnableWithMessageHistory(
  chain,  ## chain which we have created
  get_session_history,
  input_messages_key= "message"
)

In [135]:
config4 = {"configurable" : {"session_id" : "chat4"}}
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "Hi! My name is Ayush")], "language": "Hindi"},
  config= config4
)

In [136]:
response.content

'नमस्ते आयुष! मुझे आपकी मदद करने में खुशी हो रही है।  आप मुझे क्या पूछना चाहते हैं? \n'

In [138]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "What is my name?")], "language": "Hindi"},
  config= config4
)
response.content

'आपका नाम आयुष है। 😊 \n'

## Managing Conversation History:
### If not managed, messsages will grow unbounded and potentially overfolw the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

### trim_messages: Helper to Reduce how many messages we're sending to the model. this trimmer allows us to specify how mant tokens we want to keep, along with other paramters like if we want to always keep the system Message and whether to allow partial messages. 

In [146]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",          # focus on last conversation
    token_counter=llm_groq,   # your LLM for counting tokens
    include_system=True,      # always include system message
    # allow_partials=False,     # don't cut a message in half  : NOT SUPPORTED ANYMORE
    start_on="human",         # trimming starts from last HumanMessage
)

list_messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="Hi, can you tell me a story about a dragon?"),
    AIMessage(content="Sure! Once upon a time, there was a brave dragon living in the mountains."),
    HumanMessage(content="Make it funny and short."),
    AIMessage(content="The dragon sneezed fire on his own tail, and villagers laughed."),
    HumanMessage(content="Now add a knight and a princess."),
    AIMessage(content="A clumsy knight tried to save the princess, but she was already riding the dragon.")
]

# Apply trimming
trimmed = trimmer.invoke(list_messages)

In [147]:
trimmed

[SystemMessage(content='You are a helpful AI assistant.'),
 HumanMessage(content='Make it funny and short.'),
 AIMessage(content='The dragon sneezed fire on his own tail, and villagers laughed.'),
 HumanMessage(content='Now add a knight and a princess.'),
 AIMessage(content='A clumsy knight tried to save the princess, but she was already riding the dragon.')]

In [181]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
  RunnablePassthrough.assign(messages= itemgetter("message") | trimmer) | prompt | llm_groq
)   

chain.invoke(
  {
    "message":  list_messages + [HumanMessage(content= "What was the story about?")],
    "language" : "English"
  }
)

AIMessage(content="The story is a humorous take on a classic dragon tale.  \n\nIt subverts expectations by showing a dragon who is more silly than fearsome (sneezing fire on himself!), and a princess who chooses to befriend a dragon instead of needing saving by a knight. The knight's clumsiness adds to the comedic effect.  \n\n", response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 142, 'total_tokens': 215, 'completion_time': 0.132727273, 'prompt_time': 0.003526337, 'queue_time': 0.251014662, 'total_time': 0.13625361}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-6c2da016-4c9d-4ad7-931e-c6b87654923a-0', usage_metadata={'input_tokens': 142, 'output_tokens': 73, 'total_tokens': 215})

In [174]:
## Let's wrap this in the Message History
with_message_history = RunnableWithMessageHistory(
  chain,  ## chain which we have created : chain = (RunnablePassthrough.assign(messages= itemgetter("message") | trimmer) | prompt | llm_groq)
  get_session_history,
  input_messages_key= "message"
)

config5 = {"configurable" : {"session_id" : "chat5"}}

In [175]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "Hi! My name is Ayush. You know Sydney Sweeney?")], "language": "English"},
  config= config5
)
response.content

"Hi Ayush! It's nice to meet you. 😊  \n\nYes, I know Sydney Sweeney! She's a very talented actress.  What about her interests you? Have you seen any of her shows or movies? 😄 \n"

In [176]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "tell about her soap she is selling.")], "language": "English"},
  config= config5
)
response.content

'You\'re probably thinking of Sydney Sweeney\'s partnership with the skincare brand "About You". \n\nShe\'s not selling traditional soap, but "About You" offers skincare and body care products. Sydney is involved in developing and marketing these products, often sharing them with her fans online.  \n\n'

In [177]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "What actress I was talking?")], "language": "English"},
  config= config5
)
response.content

'You were talking about Sydney Sweeney! 😊  \n\nIs there anything else you\'d like to know about her or the "About You" brand?\n'

In [178]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "What is my name?")], "language": "English"},
  config= config5
)
response.content

'Your name is Ayush. 😊 I remember!   What can I do for you today? \n'

In [182]:
response = with_message_history.invoke(
  {"message" : [HumanMessage(content= "Which product of her I asked?")], "language": "English"},
  config= config5
)
response.content

'You asked about the soap she was selling, but it\'s actually skincare products from the brand "About You" that Sydney Sweeney partners with.  \n\nDo you want to know more about "About You" or Sydney Sweeney\'s involvement with them?\n'

### Ultimately, create a 'chain' and then wrap it using 'with_message_history'.